# 02 — Pembersihan dan Validasi (M2)

Aturan: salin dulu (`d = df.copy()`), **satu langkah satu catatan**, alasan ditulis di `../cleaning_log.md` saat mengerjakan. Hasil disimpan ke `../data/processed/`; `data/raw/` tidak disentuh.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# Path RELATIF terhadap folder notebooks/ (tidak ada path lokal seperti C:\Users)
RAW = Path('../data/raw/laptopPrice.csv')
PROCESSED = Path('../data/processed/laptop_bersih.csv')
FIG = Path('../reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
d = df.copy()
jejak = []

def catat(langkah, terdampak, alasan):
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'{langkah}: {int(terdampak)} baris')

print('Baris mentah:', len(df))

## Langkah 1 — Keunikan: hapus duplikat persis
Dilakukan **pertama** agar hitungan langkah berikutnya tidak terdistorsi baris ganda.

In [ ]:
n = d.duplicated().sum()
d = d.drop_duplicates().reset_index(drop=True)
catat('Hapus duplikat persis', n, 'Seluruh 19 kolom identik; satu salinan disimpan')

## Langkah 2 — Tipe data: angka bersatuan menjadi angka

In [ ]:
for kolom, baru in [('ram_gb','ram_gb'), ('ssd','ssd_gb'), ('hdd','hdd_gb'), ('graphic_card_gb','gpu_gb')]:
    d[baru] = pd.to_numeric(d[kolom].str.replace(' GB', '', regex=False), errors='raise').astype(int)
    if baru != kolom: d = d.drop(columns=kolom)
catat('Konversi ram/ssd/hdd/gpu dari teks ke angka (GB)', len(d), "Teks '8 GB' tidak dapat dihitung; satuan dipindah ke nama kolom")

d['garansi_tahun'] = d['warranty'].map({'No warranty': 0, '1 year': 1, '2 years': 2, '3 years': 3})
assert d['garansi_tahun'].notna().all()
d = d.drop(columns='warranty')
catat('Konversi warranty ke tahun (angka)', len(d), "'No warranty' = 0 tahun; nilai tak dikenal akan memicu assert")

d['rating_bintang'] = pd.to_numeric(d['rating'].str.extract(r'(\d)')[0])
d = d.drop(columns='rating')
catat('Konversi rating dari teks ke angka 1-5', len(d), "'4 stars' menjadi 4")

for k in ['Touchscreen', 'msoffice']:
    d[k.lower()] = d[k].map({'Yes': True, 'No': False})
    assert d[k.lower()].notna().all()
    if k != k.lower(): d = d.drop(columns=k)
d = d.rename(columns={'Price': 'harga', 'Number of Ratings': 'jumlah_rating', 'Number of Reviews': 'jumlah_ulasan'})
catat('Ubah Yes/No menjadi boolean dan seragamkan nama kolom', len(d), 'Boolean memudahkan agregasi; nama kolom konsisten huruf kecil')

## Langkah 3 — Konsistensi: seragamkan penulisan merek

In [ ]:
sebelum = d['brand'].nunique()
d['brand'] = d['brand'].str.strip().replace({'acer': 'Acer', 'DELL': 'Dell', 'APPLE': 'Apple'})
catat('Seragamkan penulisan merek', d['brand'].isin(['Acer','Dell','Apple']).sum(), 'Hanya huruf besar/kecil; ASUS/HP/MSI dibiarkan sebagai singkatan')
print(sorted(d['brand'].unique()), '| unik sebelum/sesudah:', sebelum, d['brand'].nunique())

## Langkah 4 — Validitas: placeholder menjadi kosong, anomali ditandai

In [ ]:
tanpa_penilai = d['jumlah_rating'] == 0
catat('Rating pada baris tanpa penilai menjadi kosong', tanpa_penilai.sum(), "Semua baris tanpa penilai berisi '3 stars' (bawaan), bukan penilaian nyata")
d.loc[tanpa_penilai, 'rating_bintang'] = np.nan

na_gen = d['processor_gnrtn'] == 'Not Available'
d['generasi_tersedia'] = ~na_gen
d.loc[na_gen, 'processor_gnrtn'] = np.nan
catat("processor_gnrtn 'Not Available' menjadi kosong + penanda", na_gen.sum(), 'Kosong terstruktur (Ryzen/Celeron/Pentium), bukan acak; diberi penanda, tidak diisi')

d['flag_32bit_ram_besar'] = (d['os_bit'] == '32-bit') & (d['ram_gb'] >= 8)
catat('Tandai OS 32-bit dengan RAM >= 8 GB', d['flag_32bit_ram_besar'].sum(), 'Belum diketahui kolom mana yang keliru; ditandai, tidak ditimpa')

q1, q3 = d['harga'].quantile([.25, .75]); batas = q3 + 1.5 * (q3 - q1)
d['harga_outlier_iqr'] = d['harga'] > batas
catat('Tandai outlier harga (IQR)', d['harga_outlier_iqr'].sum(), 'Didominasi laptop berspesifikasi tinggi (RAM 16 GB pada 38 dari 49 baris, Core i7/i9, Apple M1) sehingga wajar; dibuang akan menghapus segmen premium')

## Validasi sebelum–sesudah

In [ ]:
print('Baris           :', len(df), '->', len(d))
print('Duplikat persis :', df.duplicated().sum(), '->', d.duplicated().sum())
print('Rating rata-rata:', round(df['rating'].str.extract(r'(\d)')[0].astype(int).mean(), 2), '->', round(d['rating_bintang'].mean(), 2))
print('Kosong sesudah  :'); print(d.isna().sum()[d.isna().sum() > 0])
assert d['harga'].gt(0).all() and d.duplicated().sum() == 0
d.dtypes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
sns.countplot(x=df['rating'].str.extract(r'(\d)')[0].astype(int), ax=ax[0]); ax[0].set_title('Rating SEBELUM')
sns.countplot(x=d['rating_bintang'].dropna().astype(int), ax=ax[1]); ax[1].set_title('Rating SESUDAH')
plt.tight_layout(); plt.savefig(FIG / 'cleaning_rating_sebelum_sesudah.png', dpi=120); plt.show()

## Simpan hasil

In [ ]:
d.to_csv(PROCESSED, index=False)
print('Tersimpan:', PROCESSED, d.shape)
pd.DataFrame(jejak)